### Leer todos los datos que son requediros

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
movies_df = spark.read.table("movie_silver.movies") \
    .filter(f"file_date = '{v_file_date}'")

In [0]:
genres_df = spark.read.table("movie_silver.genres")

In [0]:
movies_genres_df = spark.read.table("movie_silver.movies_genres") \
    .filter(f"file_date = '{v_file_date}'")

### Join "genres" y "movies_genres"

In [0]:
genres_mov_gen_df = genres_df.join(movies_genres_df,
                                   genres_df.genre_id == movies_genres_df.genre_id,
                                   "inner") \
                    .select(genres_df.genre_name,
                            movies_genres_df.movie_id)

### Join "movies_df" y "genres_mov_gen_df"

- Filtrar las películas donde su fecha de lanzamiento sea mayor o igual a 2015

In [0]:
movies_filter_df = movies_df.filter("year_release_date >= 2015")

In [0]:
results_movies_genres_df = movies_filter_df.join(genres_mov_gen_df,
                                                 movies_filter_df.movie_id == genres_mov_gen_df.movie_id,
                                                 "inner")

In [0]:
results_df = results_movies_genres_df.select("year_release_date", "genre_name", "budget", "revenue")

In [0]:
from pyspark.sql.functions import sum

results_group_by_df = results_df \
    .groupBy("year_release_date", "genre_name") \
    .agg(
        sum("budget").alias("total_budget"),
        sum("revenue").alias("total_revenue")
    )

In [0]:
from pyspark.sql.functions import lit

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, dense_rank, asc

results_dense_rank_df = Window.partitionBy("year_release_date").orderBy(
                                                                desc("total_budget"), 
                                                                desc("total_revenue"))
final_df = results_group_by_df.withColumn("dense_rank", dense_rank().over(results_dense_rank_df)) \
    .withColumn("created_date", lit(v_file_date))

### Escribir datos en el DataLake en formato "Delta"

In [0]:
#overwrite_partition("movie_gold", "results_group_movie_genre", "created_date", v_file_date)

In [0]:
merge_condition = 'tgt.year_release_date = src.year_release_date AND tgt.genre_name = src.genre_name AND tgt.created_date = src.created_date'
merge_delta_lake(final_df, "movie_gold", "results_group_movie_genre", merge_condition, "created_date")

In [0]:
%sql
SELECT * FROM movie_gold.results_group_movie_genre;

year_release_date,genre_name,total_budget,total_revenue,dense_rank,created_date
2015,Adventure,2219000000,8.016412217E9,1,2024-12-30
2015,Action,2005400000,7.627777397E9,2,2024-12-30
2015,Drama,1951800000,6.473933286E9,3,2024-12-30
2015,Comedy,1623200000,6.131404237E9,4,2024-12-30
2015,Thriller,1421150000,5.617490431E9,5,2024-12-30
2015,Science Fiction,1213425000,4.333796155E9,6,2024-12-30
2015,Family,1056350000,4.235493125E9,7,2024-12-30
2015,Crime,926800000,1.961292814E9,8,2024-12-30
2015,Animation,721000000,3.532697782E9,9,2024-12-30
2015,Fantasy,576000000,1.415093823E9,10,2024-12-30
